# 🇹🇳 Stage 2 — SFT (instruction tuning) on top of CPT + TunBench eval

Continues training the **CPT adapter** on real instruction/conversation data so the model follows
instructions and answers in Tunisian Arabizi. Adds ~15% English retention (anti-forgetting), then
evaluates with the honest metrics (real-word rate + comprehension), including BASE-vs-YOURS.

## Setup
1. **+ Add Data** → upload: the **`tunisian_cpt`** folder (from Stage 1) as a Dataset,
   plus `sft_real.jsonl`, `eval_set.jsonl`, `lexicon.jsonl`, `tunbench_comprehension.jsonl`.
2. Accelerator **GPU T4 x2**, Internet **ON** (retention set + base model download).
3. **Run All** (~4–6 h). Download `/kaggle/working/tunisian_final`.


## 1. Install

In [ ]:
import importlib.metadata as _md
TARGET_TF = '4.51.3'      # Kaggle's default transformers can be broken/mismatched
try: _cur = _md.version('transformers')
except Exception: _cur = None
print('transformers found:', _cur)
if _cur != TARGET_TF:
    !pip uninstall -y -q transformers tokenizers
    !pip install -q 'transformers==4.51.3' 'peft>=0.12' 'bitsandbytes>=0.43' accelerate datasets
    print('*** INSTALLED - KERNEL IS RESTARTING. When it stops, click Run All again. ***')
    import IPython; IPython.Application.instance().kernel.do_shutdown(True)
else:
    !pip install -q 'peft>=0.12' 'bitsandbytes>=0.43' accelerate datasets
    print('env OK - continuing')


## 2. Config

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # ONE GPU: T4x2 model-split breaks the loss
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # stop VRAM fragmentation
MODEL_NAME   = 'Qwen/Qwen2.5-7B-Instruct'
CPT_ADAPTER  = 'tunisian_cpt'     # folder name of the Stage-1 dataset you uploaded
MAX_SEQ_LEN  = 1024
EPOCHS       = 1                  # 1-2; watch for overfit (loss should NOT start near 0.1)
LR           = 1e-4              # gentle on top of CPT
BATCH        = 1                  # 152k vocab -> keep logits small
GRAD_ACCUM   = 16                 # effective batch stays 16
RETENTION_N  = 2500              # English instructions mixed in to keep general ability
SYSTEM = ('Enti \'7adethni\', musa3ed tounsi. Tefhem el arabizi, el 3arbi, el français w el anglais, '
          'w tjaweb DIMA bel derja tounsiya bel arabizi (7ourouf w arqam) b tari9a tabi3iya. '
          'Ken el user yotlob 7aja b lugha o5ra, 3awenou ama 5alli el asas tounsi.')
print('SFT config ready')


## 3. Load real SFT + English retention, chat-format

In [ ]:
import json, glob, random
random.seed(42)
def find(name):
    h = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    if not h: raise FileNotFoundError(f'{name} not found under /kaggle/input')
    return h[0]
sft = [json.loads(l) for l in open(find('sft_real.jsonl'), encoding='utf-8') if l.strip()]
rows = [{'instruction': r['instruction'], 'output': r['output']} for r in sft]
# retention: keep general EN ability (prevents the reasoning collapse from last run)
try:
    from datasets import load_dataset
    al = load_dataset('tatsu-lab/alpaca', split='train').shuffle(seed=42).select(range(RETENTION_N))
    for r in al:
        instr = r['instruction'] + (('\n' + r['input']) if r['input'] else '')
        rows.append({'instruction': instr, 'output': r['output']})
    print('added retention:', RETENTION_N)
except Exception as e:
    print('retention skipped (no internet?):', e)
random.shuffle(rows)
print('total SFT rows:', len(rows))


## 4. Load base 4-bit + the CPT adapter as TRAINABLE (continue the same adapter)

In [ ]:
import gc, torch
# free any model left on the GPU by a previous run (Jupyter keeps it alive)
for _v in ['trainer','model','base','packed','train_ds']:
    if _v in globals(): del globals()[_v]
gc.collect(); torch.cuda.empty_cache()
print(f'free VRAM before load: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB of {torch.cuda.mem_get_info()[1]/1e9:.1f} GB')
assert torch.cuda.mem_get_info()[0]/1e9 > 10, 'GPU not empty -> Run > Factory reset, then Run All'
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, prepare_model_for_kbit_training
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb,
                                            device_map={'': 0}, torch_dtype=torch.float16)
base = prepare_model_for_kbit_training(base)
# continue training the CPT adapter (Stage1 -> Stage2 on ONE adapter: simple + memory-friendly)
model = PeftModel.from_pretrained(base, find(CPT_ADAPTER), is_trainable=True)
model.config.use_cache = False
model.print_trainable_parameters()


## 5. Mask the prompt, train only on the answer (DataCollator)

In [ ]:
from datasets import Dataset
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
def fmt(r):
    msgs = [{'role':'system','content':SYSTEM},{'role':'user','content':r['instruction']}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    full   = prompt + r['output'] + tokenizer.eos_token
    p_ids  = tokenizer(prompt, add_special_tokens=False)['input_ids']
    f_ids  = tokenizer(full,   add_special_tokens=False, truncation=True, max_length=MAX_SEQ_LEN)['input_ids']
    labels = [-100]*len(p_ids) + f_ids[len(p_ids):]      # supervise only the response
    return {'input_ids': f_ids, 'attention_mask':[1]*len(f_ids), 'labels': labels[:len(f_ids)]}
train_ds = Dataset.from_list(rows).map(fmt, remove_columns=['instruction','output'])
collator = DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100)
args = TrainingArguments(
    output_dir='sft_out', per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS, learning_rate=LR, warmup_ratio=0.03, logging_steps=25,
    fp16=True, optim='paged_adamw_8bit', weight_decay=0.01, lr_scheduler_type='cosine',
    gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False},
    save_strategy='no', report_to='none', seed=42, label_names=['labels'], remove_unused_columns=False)
Trainer(model=model, args=args, train_dataset=train_ds, data_collator=collator).train()


## 6. Save the final adapter

In [ ]:
model.save_pretrained('/kaggle/working/tunisian_final')
tokenizer.save_pretrained('/kaggle/working/tunisian_final')
print('saved -> /kaggle/working/tunisian_final  (point serving/ at this)')


## 7. TunBench — honest eval (real-word rate + comprehension), BASE vs YOURS

In [ ]:
import json, re
model.eval()
def gen(user, adapter=True, n=200):
    if not adapter:
        with model.disable_adapter():
            return _gen(user, n)
    return _gen(user, n)
def _gen(user, n):
    msgs=[{'role':'system','content':SYSTEM},{'role':'user','content':user}]
    ids=tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to('cuda')
    out=model.generate(input_ids=ids, attention_mask=torch.ones_like(ids), max_new_tokens=n,
        do_sample=True, temperature=0.7, top_p=0.9, repetition_penalty=1.1, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()
# --- real-word rate vs lexicon (self-contained) ---
def _norm(t): t=t.lower(); t=re.sub(r"[^\w'89]",'',t); return re.sub(r'(.)\1{2,}',r'\1',t).replace('sh','ch').replace('kh','5')
def _skel(k): return re.sub(r"[aeiouy']",'',k)
V=set(); 
for l in open(find('lexicon.jsonl'),encoding='utf-8'):
    e=json.loads(l)
    for v in (e.get('arabizi_variants') or []): V.add(_norm(v))
    for w in re.split(r'\s+', e.get('example_arabizi') or ''):
        k=_norm(w); 
        if len(k)>=2: V.add(k)
SK={_skel(v) for v in V if len(_skel(v))>=3}
W=re.compile(r"[A-Za-z0-9']+")
def realword(t):
    toks=[w for w in W.findall(t or '') if not w.isdigit()]
    if not toks: return 1.0
    ok=sum(1 for w in toks if _norm(w) in V or (len(_skel(_norm(w)))>=3 and _skel(_norm(w)) in SK) or len(_norm(w))<2)
    return ok/len(toks)
ev=[json.loads(l) for l in open(find('eval_set.jsonl'),encoding='utf-8') if l.strip()]
import statistics
base_p=[gen(r['instruction'], adapter=False) for r in ev]
new_p =[gen(r['instruction'], adapter=True)  for r in ev]
print(f'REAL-WORD RATE  base {statistics.mean(map(realword,base_p)):.0%}  ->  yours {statistics.mean(map(realword,new_p)):.0%}')
print('\n--- 8 base vs yours ---')
for r,b,a in list(zip(ev,base_p,new_p))[:8]:
    print('Q :',r['instruction'][:70]); print('base:',b[:88]); print('YOU :',a[:88]); print()


## 7b. Number-rule + slang comprehension, and dump predictions for local TunBench

In [ ]:
# number-rule conformance (valid Arabizi digits only: 2 3 5 7 8 9)
VALID=set('235789')
def num_ok(t):
    for tok in re.findall(r"[A-Za-z0-9']+", t or ''):
        if tok.isdigit() or tok.isalpha(): continue
        if any(c.isdigit() and c not in VALID for c in tok): return False
    return True
print('NUMBER-RULE   base %2.0f%% -> yours %2.0f%%' % (100*sum(map(num_ok,base_p))/len(base_p), 100*sum(map(num_ok,new_p))/len(new_p)))

# slang comprehension on the held-out test (EN gloss -> expected Arabizi word)
try:
    comp=[json.loads(l) for l in open(find('tunbench_comprehension.jsonl'),encoding='utf-8') if l.strip()][:120]
    def contains_expected(out, exps):
        ot={_norm(w) for w in re.findall(r"[A-Za-z0-9']+", out or '')}
        osk={_skel(t) for t in ot if len(_skel(t))>=2}
        return any(_norm(v) in ot or (len(_skel(_norm(v)))>=2 and _skel(_norm(v)) in osk) for v in exps)
    # short answers: cap at 48 new tokens (200 would take hours on a T4 for 240 gens)
    def comp_acc(ad): return sum(contains_expected(gen(r['instruction'], adapter=ad, n=48), r['expected']) for r in comp)/len(comp)
    print('COMPREHENSION base %2.0f%% -> yours %2.0f%%  (n=%d held-out slang)' % (100*comp_acc(False),100*comp_acc(True),len(comp)))
except FileNotFoundError:
    print('(upload tunbench_comprehension.jsonl for the comprehension score)')

# dump YOUR predictions for the full local scorer: python dataset/tools/tunbench.py score preds.jsonl
with open('/kaggle/working/preds.jsonl','w',encoding='utf-8') as f:
    for r,a in zip(ev,new_p): f.write(json.dumps({'instruction':r['instruction'],'output':a},ensure_ascii=False)+'\n')
print('dumped -> /kaggle/working/preds.jsonl  (download it, then run tunbench score locally)')


## 8. Read it
- **YOURS real-word rate ≥ base AND answers read more natural** → it worked, ship `tunisian_final`.
- If YOURS < base → the SFT hurt; ship base+grounding, and grow real conversational data.
- Then run the full **TunBench** (`dataset/tools/tunbench.py`) locally on a saved predictions file
  for comprehension + number-rule scores, and do the native A/B before shipping.
